In [1]:
import os
from pathlib import Path
import logging
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format= '%(message)s'
)

In [2]:
# Пути к файлам
rsd_path = Path("data/raw/raw_shift_data.xlsx")
dict_path = Path("data/dict/dictionary.xlsx")
clean_path = Path("data/clean/shift_flights.csv")

# Списки столбцов
check_in_col = ['check_in_agent_1', 'check_in_agent_2', 'check_in_agent_3', 'check_in_agent_4']
boarding_col = ['boarding_agent_1', 'boarding_agent_2']
desk_col = ['check_in_desk_1', 'check_in_desk_2', 'check_in_desk_3', 'check_in_desk_4']

# Проверка файлов
if rsd_path.exists():
    logging.info("Файл raw найден: %s", rsd_path)
else:
    logging.error("Файл raw не найден: %s", rsd_path)

if dict_path.exists():
    logging.info("Словарь найден: %s", dict_path)
else:
    logging.error("Словарь не найден: %s", dict_path)

logging.info("Путь для сохранения: %s", clean_path)

Файл raw найден: data\raw\raw_shift_data.xlsx
Словарь найден: data\dict\dictionary.xlsx
Путь для сохранения: data\clean\shift_flights.csv


In [3]:
def load_data():
    df=pd.read_excel(rsd_path)
    logging.info("Строк: %s, столбцов: %s", df.shape[0], df.shape[1])
    return df

df = load_data()
df.columns.tolist()

Строк: 696, столбцов: 20


['id',
 'shift_id',
 'shift_start',
 'shift_end',
 'shift_type',
 'airline_id',
 'flight_number',
 'destination_id',
 'flight_type_id',
 'aircraft_type_id',
 'dcs_id',
 'departure_time',
 'check_in_start',
 'check_in_close',
 'boarding_start',
 'pax_total',
 'check_in_agents',
 'boarding_agents',
 'check_in_desks',
 'shift_handover']

In [4]:
def load_dict():
    dict_df=pd.read_excel(dict_path)
    logging.info("Строк: %s, столбцов: %s", dict_df.shape[0], dict_df.shape[1])
    return dict_df

dict_df = load_dict()
dict_df.columns.tolist()

Строк: 56, столбцов: 2


['anon_code', 'surname']

In [5]:
# Разделение столбцов
def split_cols(df, col, new_cols):
    split = df[col].str.split(';', expand=True)
    split.columns = new_cols
    logging.info("Столбец %s разделён. Колонок: %s" , col, len(new_cols))
    return split

check_in_agents_split = split_cols(df, 'check_in_agents', check_in_col)
boarding_agents_split = split_cols(df, 'boarding_agents', boarding_col)
desks_split = split_cols(df, 'check_in_desks', desk_col)

Столбец check_in_agents разделён. Колонок: 4
Столбец boarding_agents разделён. Колонок: 2
Столбец check_in_desks разделён. Колонок: 4


In [6]:
# Удаляю исходные столбцы и вставляю разделенные
df = df.drop(['check_in_agents', 'boarding_agents', 'check_in_desks'], axis=1)
df = pd.concat([df, check_in_agents_split, boarding_agents_split, desks_split], axis=1)
df.columns.tolist()

['id',
 'shift_id',
 'shift_start',
 'shift_end',
 'shift_type',
 'airline_id',
 'flight_number',
 'destination_id',
 'flight_type_id',
 'aircraft_type_id',
 'dcs_id',
 'departure_time',
 'check_in_start',
 'check_in_close',
 'boarding_start',
 'pax_total',
 'shift_handover',
 'check_in_agent_1',
 'check_in_agent_2',
 'check_in_agent_3',
 'check_in_agent_4',
 'boarding_agent_1',
 'boarding_agent_2',
 'check_in_desk_1',
 'check_in_desk_2',
 'check_in_desk_3',
 'check_in_desk_4']

In [7]:
dict_df = pd.read_excel(dict_path)
agent_dict = dict(zip(dict_df['surname'], dict_df['anon_code']))

In [8]:
# Замена фамилий на коды с помощью словаря
def anonymize(df, agent_dict, columns):
    for col in columns:
        df[col] = df[col].map(agent_dict)

    total = df[columns].notna().sum().sum()
    logging.info("Анонимизировано значений: %s", total)

    invalid = set(df[columns].stack().dropna().unique()) - set(agent_dict.values())
    if invalid:
        logging.warning("Найдены незаменённые значения: %s", invalid)
    else:
        logging.info("Все значения анонимизированы")

    return df
    
df = anonymize(df, agent_dict, check_in_col)
df = anonymize(df, agent_dict, boarding_col)

Анонимизировано значений: 1481
Все значения анонимизированы
Анонимизировано значений: 658
Все значения анонимизированы


In [9]:
print(df[['check_in_agent_1', 'check_in_agent_2', 'check_in_agent_3', 'check_in_agent_4',
          'boarding_agent_1', 'boarding_agent_2',
          'check_in_desk_1', 'check_in_desk_2', 'check_in_desk_3', 'check_in_desk_4']].head())

  check_in_agent_1 check_in_agent_2 check_in_agent_3 check_in_agent_4  \
0              NaN              NaN              NaN              NaN   
1             A037             A049              NaN              NaN   
2             A056             A017              NaN              NaN   
3             A037             A049              NaN              NaN   
4             A036             A048              NaN              NaN   

  boarding_agent_1 boarding_agent_2 check_in_desk_1 check_in_desk_2  \
0             A054             A001             NaN             NaN   
1             A018              NaN              31              32   
2             A003              NaN              50              51   
3             A049              NaN              29              30   
4             A024              NaN              50              51   

  check_in_desk_3 check_in_desk_4  
0             NaN             NaN  
1            None            None  
2            None         

In [10]:
df.to_csv(clean_path, index=False)
logging.info("Файл сохранён: %s", clean_path)

Файл сохранён: data\clean\shift_flights.csv
